# A/B Testing & Experimentation Analysis
*Designing and analyzing product experiments: power analysis, hypothesis testing, SRM checks, and ship/no-ship recommendations — on fully synthetic data.*

**The three experiments**
| Experiment | Metric | Expected outcome |
|---|---|---|
| `exp1_checkout_cta` | Conversion rate (binary) | Variant wins |
| `exp2_onboarding` | Activation rate (binary) | No difference (null) |
| `exp3_pricing_page` | Revenue per visitor (continuous) | Variant wins, but watch the SRM check |

A null result is a result: experiment 2 is reported honestly, not forced into significance.

In [1]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# resolve repo root regardless of the kernel's working directory
_cwd = os.getcwd()
REPO = _cwd if os.path.isdir(os.path.join(_cwd, 'src')) else os.path.abspath(os.path.join(_cwd, '..'))
sys.path.insert(0, os.path.join(REPO, 'src'))
IMGDIR = os.path.join(REPO, 'images')
DATA = os.path.join(REPO, 'data', 'experiments.csv')
os.makedirs(IMGDIR, exist_ok=True)

from data_cleaning import load_and_clean
from analysis import wilson_ci, conversion_summary
from experiment import (analyze, required_n_per_arm, power_curve,
                        srm_check, ALPHA, ALPHA_ADJ, EXPERIMENTS)

sns.set_theme(style='whitegrid')

df, report = load_and_clean(DATA)
print('rows:', report['rows_clean'], '| missing:', report['missing_values'],
      '| dupes removed:', report['duplicate_rows_removed'])
df.head(3)

rows: 60000 | missing: 0 | dupes removed: 0


,experiment_id,user_id,variant,converted,revenue,assignment_date
0,exp1_checkout_cta,exp1_checkout_cta-012628,variant,0,0.0,2026-10-05
1,exp2_onboarding,exp2_onboarding-017730,variant,0,0.0,2026-10-04
2,exp2_onboarding,exp2_onboarding-019991,variant,0,0.0,2026-10-01


## 1. Experiment design & power analysis
Before looking at results: was each test adequately powered? For a 12% baseline conversion and a 5% relative minimum detectable effect (MDE), 80% power at alpha=0.05:

In [2]:
n_needed = required_n_per_arm(0.12, 0.05, power=0.8)
print(f'Required per arm for 5% relative MDE on 12% baseline: {n_needed:,} users')
print('Actually run: 10,000 users/arm/experiment')
print('-> The design reliably detects lifts >= ~11% relative; smaller true effects may be missed.')

n_range = np.linspace(2000, 60000, 60).astype(int)
fig, ax = plt.subplots(figsize=(8, 4.5))
for mde, ls in [(0.05, '--'), (0.10, '-'), (0.15, ':')]:
    ax.plot(n_range, power_curve(0.12, mde, n_range), ls=ls, label=f'MDE {mde:.0%} relative')
ax.axhline(0.8, color='gray', lw=1)
ax.axvline(10000, color='red', lw=1, ls='--', label='actual n/arm (10k)')
ax.set(xlabel='Users per arm', ylabel='Statistical power',
       title='Power curve (12% baseline, alpha=0.05)')
ax.legend(); fig.tight_layout()
fig.savefig(os.path.join(IMGDIR, 'power_curve.png'), dpi=150); plt.close(fig)
print('saved images/power_curve.png')

Required per arm for 5% relative MDE on 12% baseline: 46,047 users
Actually run: 10,000 users/arm/experiment
-> The design reliably detects lifts >= ~11% relative; smaller true effects may be missed.


saved images/power_curve.png


## 2. Data quality & EDA
Sanity checks first: variant splits (SRM), then conversion behavior.

In [3]:
# SRM check per experiment
for exp in EXPERIMENTS:
    sub = df[df.experiment_id == exp]
    nc = (sub.variant == 'control').sum(); nv = (sub.variant == 'variant').sum()
    _, p = srm_check(nc, nv)
    flag = 'OK  ' if p >= ALPHA else 'FLAG'
    print(f'{exp}: control={nc:,} variant={nv:,}  SRM p={p:.4f} [{flag}]')

# conversion bars with Wilson 95% CI (binary experiments)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
for ax, exp in zip(axes, ['exp1_checkout_cta', 'exp2_onboarding']):
    s = conversion_summary(df, exp)
    labels = ['control', 'variant']
    rates = [s[v]['rate'] for v in labels]
    lo = [s[v]['rate'] - s[v]['ci_lo'] for v in labels]
    hi = [s[v]['ci_hi'] - s[v]['rate'] for v in labels]
    ax.bar(labels, rates, yerr=[lo, hi], capsize=6, color=['#6b7c93', '#2E9CD3'])
    ax.set(title=exp.replace('_', ' '), ylabel='conversion rate')
    for i, r in enumerate(rates):
        ax.text(i, r + hi[i] + 0.004, f'{r:.2%}', ha='center', fontsize=10)
fig.suptitle('Conversion rate by variant (Wilson 95% CI)')
fig.tight_layout()
fig.savefig(os.path.join(IMGDIR, 'conversion_ci.png'), dpi=150); plt.close(fig)
print('saved images/conversion_ci.png')

exp1_checkout_cta: control=10,000 variant=10,000  SRM p=1.0000 [OK  ]
exp2_onboarding: control=10,000 variant=10,000  SRM p=1.0000 [OK  ]
exp3_pricing_page: control=10,150 variant=9,850  SRM p=0.0339 [FLAG]


saved images/conversion_ci.png


In [4]:
# revenue distribution for exp3 (positive revenues, log scale)
sub3 = df[df.experiment_id == 'exp3_pricing_page']
fig, ax = plt.subplots(figsize=(8, 4.5))
for variant, color in [('control', '#6b7c93'), ('variant', '#2E9CD3')]:
    rev = sub3[(sub3.variant == variant) & (sub3.revenue > 0)].revenue
    ax.hist(rev, bins=60, alpha=0.55, label=f'{variant} (n={len(rev):,})', color=color)
ax.set_xscale('log')
ax.set(xlabel='revenue per paying visitor ($) [log scale]', ylabel='visitors',
       title='exp3_pricing_page: revenue distribution by variant')
ax.legend(); fig.tight_layout()
fig.savefig(os.path.join(IMGDIR, 'revenue_distribution.png'), dpi=150); plt.close(fig)
m = sub3.groupby('variant').revenue.mean()
print(f"mean revenue/visitor: control=${m['control']:.2f} variant=${m['variant']:.2f}")
print('saved images/revenue_distribution.png')

mean revenue/visitor: control=$3.57 variant=$4.22
saved images/revenue_distribution.png


## 3. Results: hypothesis tests
Two-proportion z-test for the binary experiments, Welch's t-test for revenue. With 3 experiments we judge significance against the Bonferroni-adjusted alpha = 0.05/3 ≈ 0.0167.

In [5]:
results = [analyze(df, e) for e in EXPERIMENTS]
t = pd.DataFrame(results)
show = t[['experiment', 'p_value', 'significant_bonferroni', 'rel_lift', 'srm_p', 'decision']].copy()
show['p_value'] = show['p_value'].map(lambda x: f'{x:.4f}')
show['rel_lift'] = show['rel_lift'].map(lambda x: f'{x:.1%}')
show['srm_p'] = show['srm_p'].map(lambda x: f'{x:.4f}')
show

,experiment,p_value,significant_bonferroni,rel_lift,srm_p,decision
0,exp1_checkout_cta,0.0001,True,15.3%,1.0000,SHIP
1,exp2_onboarding,0.8117,False,0.5%,1.0000,NO SHIP
2,exp3_pricing_page,0.0088,True,18.1%,0.0339,DO NOT SHIP - investigate SRM


In [6]:
# p-value summary: -log10(p) vs Bonferroni threshold
fig, ax = plt.subplots(figsize=(8, 4.2))
exps = [r['experiment'].replace('_', chr(10)) for r in results]
vals = [-np.log10(r['p_value']) for r in results]
colors = ['#2ca02c' if r['significant_bonferroni'] else '#d62728' for r in results]
ax.bar(exps, vals, color=colors)
ax.axhline(-np.log10(ALPHA_ADJ), color='black', ls='--',
           label=f'Bonferroni alpha={ALPHA_ADJ:.4f}')
ax.set(ylabel='-log10(p-value)',
       title='Significance per experiment (green = significant @ Bonferroni alpha)')
ax.legend(); fig.tight_layout()
fig.savefig(os.path.join(IMGDIR, 'pvalue_summary.png'), dpi=150); plt.close(fig)
print('saved images/pvalue_summary.png')

saved images/pvalue_summary.png


In [7]:
# forest plot: relative lift with 95% CI per experiment
def rel_lift_ci(exp_id):
    sub = df[df.experiment_id == exp_id]
    c = sub[sub.variant == 'control']; v = sub[sub.variant == 'variant']
    if EXPERIMENTS[exp_id]['kind'] == 'proportion':
        pc, pv = c.converted.mean(), v.converted.mean()
        se = np.sqrt(pc * (1 - pc) / len(c) + pv * (1 - pv) / len(v))
        lo, hi = (pv - pc - 1.96 * se) / pc, (pv - pc + 1.96 * se) / pc
        return (pv - pc) / pc, lo, hi
    mc, mv = c.revenue.mean(), v.revenue.mean()
    se = np.sqrt(c.revenue.var(ddof=1) / len(c) + v.revenue.var(ddof=1) / len(v))
    lo, hi = (mv - mc - 1.96 * se) / mc, (mv - mc + 1.96 * se) / mc
    return (mv - mc) / mc, lo, hi

fig, ax = plt.subplots(figsize=(8, 4.2))
ys = np.arange(len(results))
for i, r in enumerate(results):
    est, lo, hi = rel_lift_ci(r['experiment'])
    ax.errorbar(est, ys[i], xerr=[[est - lo], [hi - est]], fmt='o',
                color='#2E9CD3', ecolor='#2E9CD3', capsize=5)
    ax.text(hi + 0.008, ys[i], f'{est:.1%}', va='center', fontsize=10)
ax.axvline(0, color='black', lw=1)
ax.axvline(0.05, color='gray', ls='--', label='MDE (5% relative)')
ax.set_yticks(ys); ax.set_yticklabels([r['experiment'] for r in results])
ax.set(xlabel='relative lift vs control (95% CI)', title='Effect sizes per experiment')
ax.legend(); fig.tight_layout()
fig.savefig(os.path.join(IMGDIR, 'lift_forest.png'), dpi=150); plt.close(fig)
print('saved images/lift_forest.png')

saved images/lift_forest.png


## 4. Recommendations

| Experiment | Decision | Why |
|---|---|---|
| exp1_checkout_cta | **SHIP** | Significant after Bonferroni; lift well above the 5% MDE; SRM clean |
| exp2_onboarding | **NO SHIP** | No detectable effect — a null result is still a result; don't ship, revisit the hypothesis |
| exp3_pricing_page | **DO NOT SHIP — investigate SRM** | Revenue lift looks real, but the SRM check flagged the split. Never ship on a broken randomization — audit assignment logging first |

In [8]:
for r in results:
    print(f"[{r['experiment']}] {r['decision']}")
    print(f"  {r['reason']}")
    print()

[exp1_checkout_cta] SHIP
  Significant (p=0.0001) with relative lift 15.3% above the MDE. Recommend shipping to 100%.

[exp2_onboarding] NO SHIP
  No significant effect (p=0.812 >= Bonferroni alpha=0.0167). Do not ship; consider iterating on the hypothesis.

[exp3_pricing_page] DO NOT SHIP - investigate SRM
  Sample-ratio mismatch detected: the variant split deviates from 50/50. Check randomization/logging before trusting any result.

